## 1. Run Combo Lock Interaction (verl)

In [1]:
import os
import sys
sys.path.append('../')

from verl_submodule.verl.interactions.combolock_interaction import ComboLockInteraction

In [2]:
interaction = ComboLockInteraction(config={})

In [3]:
instance_id = await interaction.start_interaction(
    instance_id=None,
    combination_length=3,
    max_attempts=8,
    vocab='0123456789',
    ground_truth='835',
    format='interaction_think',
)
env = interaction._instance_dict[instance_id]['env']

In [4]:
guesses = ['123', '456', '789', '835']  # example guesses
attempts = 0
game_history = []

while attempts < env.max_attempts:

    print(f"\nAttempt {attempts + 1}/{env.max_attempts}:")

    attempts += 1
    guess = guesses[attempts - 1]
    messages = [
        {"role": "user", "content": f"Make guess for {env.combination_length}-digit \combination"},
        {"role": "assistant", "content": f"<action>{guess}</action>"}
    ]
    
    # Get response from the interaction
    done, response, score, additional_data = await interaction.generate_response(
        instance_id=instance_id,
        messages=messages
    )
    
    print(f"Response: {response}")
    print(f"Score: {score}")
    
    # Record the attempt
    game_history.append({
        "attempt": attempts,
        "guess": guess,
        "response": response,
        "score": score
    })
    
    if done:
        print(f"\n🎉 Game completed! Goal reached: {done}")
        break


Attempt 1/8:
Response: 1 is not in the lock
2 is not in the lock
3 is not in Position 3, but is in the lock
Score: 0.0

Attempt 2/8:
Response: 4 is not in the lock
5 is not in Position 2, but is in the lock
6 is not in the lock
Score: 0.0

Attempt 3/8:
Response: 7 is not in the lock
8 is not in Position 2, but is in the lock
9 is not in the lock
Score: 0.0

Attempt 4/8:
Response: 8 is in Position 1!
3 is in Position 2!
5 is in Position 3!
Score: 0.625

🎉 Game completed! Goal reached: True


# 2. Run Paprika Interaction (verl) in the same way

In [5]:
import asyncio
import os
import sys
import random

# Add the verl and paprika paths to sys.path if needed
sys.path.append('../verl_submodule')
sys.path.append('../paprika')
sys.path.append('../')
from notebooks.paprika_config_helper import PaprikaConfigHelper

In [6]:
from verl.interactions.paprika_interaction import PaprikaInteraction

In [7]:
GAME_NAME = "twenty_questions"  # change to e.g. "twenty_questions", "mastermind", etc.
config = PaprikaConfigHelper.create_config(GAME_NAME)
config['belief_config']['style'] = 'none'  # no belief right now

In [8]:
config

{'game_env_name': 'twenty_questions',
 'agent_config': {'model_type': 'openai_api_models',
  'model_name': 'gpt-4o-mini',
  'model_max_length': 20000},
 'env_config': {'model_type': 'openai_api_models',
  'model_name': 'gpt-4o-mini',
  'model_max_length': 20000},
 'judge_config': {'model_type': 'openai_api_models',
  'model_name': 'gpt-4o-mini',
  'model_max_length': 1000},
 'belief_config': {'style': 'none'}}

In [9]:
def random_guess(game_name): # replace with agent when running training/inference
    if game_name == "wordle":
        import string
        return ''.join(random.choices(string.ascii_lowercase, k=5))
    elif game_name == "mastermind":
        digits = list("123456")
        random.shuffle(digits)
        return ''.join(digits[:4])
    elif game_name == "twenty_questions":
        return "Is it alive?"
    else:
        return "RANDOM_ACTION"

In [10]:
interaction = PaprikaInteraction(config={})

In [11]:
instance_id = await interaction.start_interaction(
        instance_id=None,
        **config
    )

In [12]:
game_env = config["game_env_name"]
game_environment = interaction._instance_dict[instance_id]["game_environment"]
game_scenarios = game_environment.get_game_scenarios(config={"data_type": "eval", "data_subtype": None})

In [13]:
game_environment

In [14]:
scenario = game_scenarios[0]

In [15]:
scenario

{'env': 'Gloves', 'agent': 'clothing'}

In [16]:
messages = [
        {"role": "user", "content": "Let's play!"},
        {"role": "assistant", "content": f"<action>{random_guess(game_env)}</action>"}
    ]

In [17]:
done, response, score, additional_data = await interaction.generate_response(
        instance_id=instance_id,
        messages=messages,
        scenario=scenario,
    )

The inference engine has been reset to the start state!
The inference engine has been reset to the start state!
The inference engine has been reset to the start state!


In [18]:
done, response, score

(True, 'Game completed. Goal reached: False, Turns: 20/20', 0.0)

In [21]:
additional_data['record'].keys()

dict_keys(['agent_game_scenario', 'env_game_scenario', 'goal_reached', 'judge_label', 'num_turns', 'max_turns', 'env_first_message', 'conversation', 'conversation_llm_responses', 'env_conversation', 'judge_conversation', 'rewards', 'belief_config', 'belief_actions_convs'])

In [23]:
additional_data['record']['conversation'][:10]

[{'role': 'system', 'content': 'You are a helpful assistant.'},
 {'role': 'user',
  'content': "You are playing a game of 20 Questions. Your goal is to guess the name of a thing or person by asking up to 20 yes-or-no questions. After each question, you will receive an answer: 'Yes' or 'No.' Use the answers provided to refine your guesses.\n\nHere are your instructions:\n- You can ask only yes-or-no questions.\n- After receiving each answer, you should adapt your questions based on the new information.\n- Your goal is to guess the topic in as few questions as possible.\n- If you're confident, you can make a guess before reaching 20 questions.\n\nThe game starts now. You are trying to guess a clothing. Ask your first question!"},
 {'role': 'assistant',
  'content': 'Is the clothing item primarily worn on the upper body?'},
 {'role': 'user', 'content': 'No'},
 {'role': 'assistant',
  'content': 'Is the clothing item primarily worn on the lower body?'},
 {'role': 'user', 'content': 'No'},
